#  Modèles Avancés — Sentence-BERT

**Projet 3 — Détection automatique de Fake News politiques**

Ce notebook :
- Génère des embeddings sémantiques avec `all-MiniLM-L6-v2`
- Entraîne Logistic Regression et Gradient Boosting sur les embeddings
- Compare avec les baselines TF-IDF
- Sauvegarde les embeddings et le meilleur modèle

>  Durée estimée : **~10 min** sur CPU standard

## 0. Imports & Configuration

In [ ]:
import os, json, joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, f1_score,
    accuracy_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
SEED = 42
SBERT_MODEL = 'all-MiniLM-L6-v2'

print(' Imports OK')

## 1. Chargement des données prétraitées

In [ ]:
train_df = pd.read_parquet('../data/traitees/liar_train.parquet')
val_df   = pd.read_parquet('../data/traitees/liar_val.parquet')
test_df  = pd.read_parquet('../data/traitees/liar_test.parquet')

X_train_raw = train_df['enriched_text'].fillna('').tolist()
X_val_raw   = val_df['enriched_text'].fillna('').tolist()
X_test_raw  = test_df['enriched_text'].fillna('').tolist()

y_train = train_df['binary_label'].values
y_val   = val_df['binary_label'].values
y_test  = test_df['binary_label'].values

print(f'Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}')

## 2. Génération des embeddings Sentence-BERT

> Le modèle `all-MiniLM-L6-v2` (~80 Mo) est téléchargé automatiquement au premier lancement puis mis en cache.

In [ ]:
EMB_TRAIN = '../data/traitees/sbert_train.npy'
EMB_VAL   = '../data/traitees/sbert_val.npy'
EMB_TEST  = '../data/traitees/sbert_test.npy'

print(f' Modèle : {SBERT_MODEL}')
sbert = SentenceTransformer(SBERT_MODEL)

if os.path.exists(EMB_TRAIN):
    print(' Embeddings en cache — chargement...')
    X_train_emb = np.load(EMB_TRAIN)
    X_val_emb   = np.load(EMB_VAL)
    X_test_emb  = np.load(EMB_TEST)
else:
    print(' Calcul des embeddings (~8 min sur CPU)...')
    X_train_emb = sbert.encode(X_train_raw, batch_size=64, show_progress_bar=True)
    X_val_emb   = sbert.encode(X_val_raw,   batch_size=64, show_progress_bar=True)
    X_test_emb  = sbert.encode(X_test_raw,  batch_size=64, show_progress_bar=True)
    np.save(EMB_TRAIN, X_train_emb)
    np.save(EMB_VAL,   X_val_emb)
    np.save(EMB_TEST,  X_test_emb)
    print(' Embeddings sauvegardés dans data/traitees/')

print(f'\nShape embeddings train : {X_train_emb.shape}')
print(f'Dimension vecteur SBERT : {X_train_emb.shape[1]}')

## 3. Entraînement des modèles sur embeddings SBERT

In [ ]:
MODELS_SBERT = {
    'SBERT + Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, C=1.0, random_state=SEED
    ),
    'SBERT + Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5, random_state=SEED
    )
}

results_sbert = {}
for name, model in MODELS_SBERT.items():
    print(f' {name}...')
    model.fit(X_train_emb, y_train)
    y_pred = model.predict(X_test_emb)
    results_sbert[name] = {
        'model':       model,
        'y_pred':      y_pred,
        'accuracy':    round(accuracy_score(y_test, y_pred), 4),
        'f1_macro':    round(f1_score(y_test, y_pred, average='macro'), 4),
        'f1_weighted': round(f1_score(y_test, y_pred, average='weighted'), 4),
    }
    print(f'   Accuracy={results_sbert[name]["accuracy"]:.3f} | '
          f'F1 macro={results_sbert[name]["f1_macro"]:.3f}')

best_sbert = max(results_sbert, key=lambda k: results_sbert[k]['f1_macro'])
print(f'\n🏆 Meilleur SBERT : {best_sbert}')

## 4. Comparaison TF-IDF Baselines vs SBERT

In [ ]:
# Charge les métriques des baselines
with open('../data/modeles/baselines_metrics_test.json', 'r') as f:
    baseline_metrics = json.load(f)

# Construction du tableau comparatif
rows = []
for name, m in baseline_metrics.items():
    rows.append({'Modèle': name, 'Type': 'TF-IDF',
                 'Accuracy': m['accuracy'], 'F1 Macro': m['f1_macro'],
                 'F1 Weighted': m['f1_weighted']})
for name, res in results_sbert.items():
    rows.append({'Modèle': name, 'Type': 'SBERT',
                 'Accuracy': res['accuracy'], 'F1 Macro': res['f1_macro'],
                 'F1 Weighted': res['f1_weighted']})

all_comp = pd.DataFrame(rows).sort_values('F1 Macro', ascending=False)
print('📊 Comparaison complète :\n')
display(all_comp.reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Graphique comparatif
x = np.arange(len(all_comp))
w = 0.25
colors_bar = ['#1D9E75' if t == 'SBERT' else '#7F77DD' for t in all_comp['Type']]
axes[0].bar(x-w, all_comp['Accuracy'],    w, label='Accuracy',    color='#7F77DD', alpha=0.8)
axes[0].bar(x,   all_comp['F1 Macro'],    w, label='F1 Macro',    color=colors_bar)
axes[0].bar(x+w, all_comp['F1 Weighted'], w, label='F1 Weighted', color='#EF9F27', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(all_comp['Modèle'], rotation=25, ha='right', fontsize=9)
axes[0].set_ylim(0, 1)
axes[0].set_title('TF-IDF vs Sentence-BERT', fontweight='bold')
axes[0].set_ylabel('Score')
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color='#7F77DD', label='TF-IDF'),
    Patch(color='#1D9E75', label='SBERT')
], loc='lower right')

# Matrice de confusion meilleur SBERT
cm = confusion_matrix(y_test, results_sbert[best_sbert]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['fake','real'], yticklabels=['fake','real'])
axes[1].set_title(f'{best_sbert}\nF1={results_sbert[best_sbert]["f1_macro"]:.3f}',
                  fontweight='bold')
axes[1].set_xlabel('Prédit')
axes[1].set_ylabel('Vrai label')

plt.tight_layout()
plt.savefig('../Doc/ADV_01_sbert_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f'Rapport détaillé — {best_sbert} :\n')
print(classification_report(y_test, results_sbert[best_sbert]['y_pred'],
                             target_names=['fake','real']))

## 5. Sauvegarde

In [ ]:
# Sauvegarde meilleur modèle SBERT
joblib.dump(results_sbert[best_sbert]['model'],
            '../data/modeles/sbert_best_clf.joblib')

# Métriques SBERT
sbert_metrics = {n: {k: v for k, v in res.items() if k != 'model' and k != 'y_pred'}
                 for n, res in results_sbert.items()}
with open('../data/modeles/sbert_metrics_test.json', 'w') as f:
    json.dump(sbert_metrics, f, indent=2)

# Toutes les métriques combinées
all_comp.to_json('../data/modeles/all_models_metrics.json', orient='records', indent=2)

print('💾 data/modeles/sbert_best_clf.joblib')
print('💾 data/modeles/sbert_metrics_test.json')
print('💾 data/modeles/all_models_metrics.json')
print('\n✅ Modèles avancés terminés → lancer Evaluation_Hors_Domaine.ipynb')